# 03b · Task3 fast red/green null

In [ ]:
from pathlib import Path
import sys,numpy as np,pandas as pd
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'; sys.path.insert(0,str(PIPE)); import config
from utils.epochs import load_epochs
from utils.decoding import make_time_windows,fixed_cv,decode_accuracy,decode_permutation
from utils.stats import cluster_permutation_1d
ep=load_epochs(config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/'task3_erp_clean.npz'); trig=np.char.replace(ep['triggers'].astype(str),'Trigger-In:',''); names=list(ep['channel_names'])
cand=pd.read_csv(config.subject_result_dir('test001')/'localizer'/'candidate_electrodes_10mm.csv'); col=next(c for c in cand.columns if c.lower() in ('channel','channelname','name')); picks=[names.index(c) for c in cand[col].drop_duplicates() if c in names]
mask=np.isin(trig,['51','54']); y=(trig[mask]=='54').astype(int); windows,centers=make_time_windows(ep['times_ms'],config.WINDOW_MS,config.STEP_MS); cv=fixed_cv(y,config.N_SPLITS,config.RANDOM_SEED); x=ep['data'][mask][:,picks,:]; observed=decode_accuracy(x,y,windows,cv); null=decode_permutation(x,y,windows,cv,config.N_PERMUTATIONS,config.RANDOM_SEED,config.N_JOBS); p=(1+(null>=observed).sum(0))/(1+len(null)); pc=cluster_permutation_1d(observed,null); out=config.subject_result_dir('test001')/'task3'; pd.DataFrame({'time_ms':centers,'accuracy':observed,'p_uncorrected':p,'p_cluster':pc}).to_csv(out/'red_green_decoding_fast.csv',index=False); print('permutations',config.N_PERMUTATIONS,'min_cluster_p',pc.min())